## Setup

In [1]:
#Import libs
library(ggplot2)
library(patchwork)
library(pheatmap)
library(viridis)
library(Seurat)
library(dplyr)
library(gridExtra)
library(stringi)
library(ROCit)
library(png)


Loading required package: viridisLite

Loading required package: SeuratObject

Loading required package: sp


Attaching package: 'SeuratObject'


The following objects are masked from 'package:base':

    intersect, t



Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'gridExtra'


The following object is masked from 'package:dplyr':

    combine




In [2]:
# Set working directory 
base_dir <- "/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow"

#Create results folder
res_folder <- file.path(base_dir, "results")
dir.create(res_folder, showWarnings = FALSE, recursive = TRUE) #Create results folder if it does not exist


In [ ]:
#dictionary of slide names from patients and their preparation method 
sample_mapping <- c(
  "frozen_b_1",
  "ffpe_c_51",
)

#get list of slide names
slide_list <- sample_mapping


## Data preprocessing

In [ ]:
### Data processing
# Main processing loop
for(slide in slide_list){
  print(paste("Processing slide:", slide))
  
  #retrieve slide sample ID
  sample_id <- slide
    
  # Find the GSM number for this sample
  raw_files <- list.files(path = file.path(base_dir, "GSE175540_RAW"), 
                          pattern = paste0("_", sample_id, "_filtered_feature_bc_matrix.h5$"),
                          full.names = TRUE)
  
  # Extract GSM number and construct file paths
  filtered_h5 <- raw_files[1]
  gsm_prefix <- sub("_filtered_feature_bc_matrix.h5", "", basename(filtered_h5))
  
  # Read the filtered matrix
  expression_matrix <- Seurat::Read10X_h5(filtered_h5)
  
  # Get other spatial data files
  raw_dir <- file.path(base_dir, "GSE175540_RAW")
  tissue_pos_file <- file.path(raw_dir, paste0(gsm_prefix, "_tissue_positions_list.csv.gz"))
  scale_factors_file <- file.path(raw_dir, paste0(gsm_prefix, "_scalefactors_json.json.gz"))
  hires_image_file <- file.path(raw_dir, paste0(gsm_prefix, "_tissue_hires_image.png.gz"))
  lowres_image_file <- file.path(raw_dir, paste0(gsm_prefix, "_tissue_lowres_image.png.gz"))
  
  # Create temporary directory structure for spatial files 
  temp_base_dir <- file.path(tempdir(), "spatial_temp", slide)
  temp_outs_dir <- file.path(temp_base_dir, "outs")
  temp_spatial_dir <- file.path(temp_outs_dir, "spatial")
  dir.create(temp_spatial_dir, recursive = TRUE, showWarnings = FALSE)
  
  # Copy the H5 file to temp directory
  temp_h5 <- file.path(temp_outs_dir, "filtered_feature_bc_matrix.h5")
  file.copy(filtered_h5, temp_h5, overwrite = TRUE)
  
  # Decompress spatial files to temp directory
  system(paste("gunzip -c", shQuote(tissue_pos_file), ">", file.path(temp_spatial_dir, "tissue_positions_list.csv")))
  system(paste("gunzip -c", shQuote(scale_factors_file), ">", file.path(temp_spatial_dir, "scalefactors_json.json")))
  system(paste("gunzip -c", shQuote(hires_image_file), ">", file.path(temp_spatial_dir, "tissue_hires_image.png")))
  system(paste("gunzip -c", shQuote(lowres_image_file), ">", file.path(temp_spatial_dir, "tissue_lowres_image.png")))
  
  # Load using Load10X_Spatial 
  spatial_object <- Load10X_Spatial(data.dir = temp_outs_dir, filename = "filtered_feature_bc_matrix.h5", slice = slide)
  
  # Collect all genes coded on the mitochondrial genome
  mt.genes <- grep(pattern = "^MT-", x = rownames(spatial_object), value = TRUE)
  counts_data <- GetAssayData(spatial_object, assay = "Spatial", layer = "counts")
  spatial_object$percent.mito <- (Matrix::colSums(counts_data[mt.genes, ])/Matrix::colSums(counts_data))*100
  
  #remove mt genes
  genes_to_keep <- setdiff(names(which(Matrix::rowSums(counts_data) > 5)), mt.genes)
  
  spatial_object_subset <- subset(spatial_object, features = genes_to_keep, subset = nFeature_Spatial > 300 & percent.mito < 30)
  cat("  Spots removed: ", ncol(spatial_object) - ncol(spatial_object_subset), "\n")
  cat("  Genes kept: ", length(genes_to_keep), "from", nrow(spatial_object), "\n") 
  
  spatial_object_subset <- suppressWarnings(SCTransform(spatial_object_subset, assay = "Spatial", verbose = FALSE))
  
  # Add TLS annotations to the object (single annotation file per sample)
  annot_file <- file.path(raw_dir, paste0(gsm_prefix, "_TLS_annotation.csv.gz"))
  annot_table <- read.csv(gzfile(annot_file))
  
  # Set rownames to match cell IDs
  rownames(annot_table) <- annot_table[,1]
  annot_table[,1] <- NULL  
  
  # Add metadata to the object
  spatial_object_subset <- AddMetaData(
    object = spatial_object_subset,
    metadata = annot_table,
    col.name = paste0(colnames(annot_table), "_annot")
  )
  
  cat("  Added annotations from:", basename(annot_file), "\n")
  
  # Save processed object to disk
  saveRDS(spatial_object_subset, file.path(res_folder, paste0("processed_", slide, ".rds")))
  
  cat("Finished processing", slide, "\n\n")
}

cat("\n=== Processing complete ===\n")
cat("Processed samples saved to:", res_folder, "\n")

## Analysis

In [11]:
###Retrieve processed sample data
# Get all processed RDS files
rds_files <- list.files(path = res_folder, 
                        pattern = "^processed_.*\\.rds$", 
                        full.names = TRUE)

cat("Found", length(rds_files), "processed sample files\n") 


Found 3 processed sample files


In [ ]:
#Load TLS signature genes
Literature <- list('IGHA1', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGHM', 'IGKC', 'IGLC1', 'IGLC7', 'JCHAIN', 'CD79A', 'FCRL5', 'MZB1', 'SSR4', 'XBP1', 'TRBC2', 'IL7R', 'CXCL12', 'LUM', 'C1QA', 'C7', 'CD52', 'APOE', 'PTTG1IP', 'PTGDS', 'PIM2', 'DERL3')

CD4_LGMN_Treg <- list('CD4', 'CD14', 'LGMN', 'GRN', 'SLCO2B1', 'CD68', 'STAB1', 'MSR1', 'CD163', 'CSF1R',
 'MAN2B1', 'GAA', 'SLC40A1', 'ITGB2', 'CYBA', 'CYBB', 'SIGLEC1', 'AP1B1', 'CTSL',
 'AXL', 'CPVL', 'CTSC', 'SDC3', 'FCGR3A', 'LIPA', 'NCOA4', 'CREG1', 'MAFB', 'NCF1',
 'HCK', 'SPI1', 'FUCA1', 'CIITA', 'SCPEP1', 'TBXAS1', 'MMP14', 'CTSH', 'FCGR2A',
 'LGALS9', 'LAIR1', 'THEMIS2', 'STAT1', 'F13A1', 'GAS6', 'MRC2', 'TRPM2', 'VSIG4',
 'HMOX1', 'ABCA1', 'CD300A', 'SLC7A7', 'ITGAX', 'LRP1', 'FERMT3', 'SAMHD1', 'TYMP',
 'C3AR1', 'TREM2', 'PLTP', 'CMKLR1', 'LILRB1', 'OLFML3', 'CTSA', 'NAIP', 'SGK1',
 'GLB1', 'UCP2', 'ENG', 'WAS', 'FCGR1A', 'CXCL16', 'GNAI2', 'FOLR2', 'PLAUR',
 'AOAH', 'TNFSF13', 'ADGRE2', 'ST14', 'LAMP1', 'ABI3', 'PFKFB3', 'CXCL12',
 'IFNGR1', 'MRC1', 'ZYX', 'GPR34', 'WARS', 'CCR1', 'SCIMP', 'ACP5', 'PLEKHO1',
 'CEBPA', 'ARRB2', 'CSF3R', 'GIMAP5', 'AKR1B1', 'NR1H3', 'TKT', 'NINJ1', 'PTAFR')

CXCR4_CD4_Tcell <- list('CXCR4', 'MYH9', 'SMAP2', 'CD3E', 'CD44', 'ERN1', 'SELPLG', 'EEF1G', 'LENG8',
 'TMEM173', 'ITK', 'ZAP70', 'CD5', 'FYN', 'SEMA4D', 'KCNA3', 'PRDM1', 'ACAP1',
 'IKZF3', 'FCMR', 'STAT4', 'CD4', 'TCF7', 'IKZF1', 'CD6', 'PDCD4', 'DDX39B', 'OGT',
 'SEPTIN9', 'STK4', 'PTK2B', 'PBXIP1', 'TC2N', 'GIMAP5', 'CD40LG', 'NLRP1',
 'SEPTIN6', 'ITGAL', 'PLEC', 'CD247', 'GRK2', 'PLCG1', 'SH3BP5', 'CTBP1', 'TGFBR2',
 'PRKCQ', 'IL16', 'CDC14A', 'PDE4B', 'ETS1', 'FLT3LG', 'CCR7', 'CD96', 'HNRNPH1',
 'RUNX2', 'KLRG1', 'SMAD3', 'SIGIRR', 'LY9', 'EPHA1', 'CD99', 'DIP2A', 'PTGER4',
 'FBLN5', 'CCN2', 'ECRG4', 'TNK2', 'HECA', 'TSPAN14', 'TNFSF8', 'CCR6', 'MAP4K2',
 'SORL1', 'CCR4', 'SLAMF1', 'LEF1', 'ADGRE5', 'IL11RA', 'KDM5D', 'SYNPO2',
 'SLC2A3', 'S1PR1', 'SELL', 'ABCA7', 'AEBP1', 'CCL19', 'FOXP1', 'CD28', 'NIBAN1',
 'CASP8', 'CXCR6', 'ATM', 'SATB1', 'CCDC80', 'KLRB1', 'SP140', 'EPHA4', 'SLAMF6',
 'SFRP1', 'CFLAR')

LAG3_IRF1_IFNrelated <- list('CD27', 'CST7', 'LAG3', 'IRF1', 'ITGAL', 'CD8A', 'TAP1', 'IL2RG', 'TNFSF10',
 'STAT1', 'TNFRSF9', 'SPN', 'CXCR3', 'PTPN6', 'TAP2', 'ITGB2', 'CSF1', 'CD3E',
 'GZMA', 'NLRC5', 'RASGRP1', 'GZMK', 'NTNG2', 'GBP5', 'PIK3CD', 'GBP1', 'JAK3',
 'HNRNPLL', 'IL2RB', 'IGF2R', 'APOBEC3G', 'GPR174', 'CD2', 'TRAF5', 'ITGA4',
 'SEMA4D', 'CDH6', 'ITGB7', 'CD247', 'CD8B', 'CSK', 'TOX2', 'GRK2', 'PRKCH',
 'SIRPG', 'APMAP', 'APOBEC3C', 'SH2D2A', 'IL16', 'LAT', 'IL10RA', 'VEGFA', 'LBH',
 'HDAC1', 'CD3G', 'TNFRSF14', 'IKZF3', 'CD82', 'CPT1A', 'CHD3', 'CP', 'TOX',
 'FASLG', 'ADAR', 'RASSF5', 'C4B', 'PDIA3', 'BTN3A1', 'FCRL3', 'MCTP2', 'CCR5',
 'EOMES', 'UBD', 'OGT', 'DPP4', 'KLRK1', 'TNFRSF1B', 'ATXN1', 'LCK', 'PECAM1',
 'UBASH3A', 'KMT2A', 'CIITA', 'NCOR2', 'UCP2', 'WARS', 'CD38', 'HNRNPH1', 'HNRNPD',
 'RASSF1', 'ADGRE5', 'IL21R', 'GTF2I', 'ITM2A', 'CD96', 'DNM2', 'HAVCR2', 'ATXN2L',
 'PRF1', 'GUSB')

# Define the signature gene sets
signatures <- list(
  Literature = Literature,
  CD4_LGMN_Treg = CD4_LGMN_Treg,
  CXCR4_CD4_Tcell = CXCR4_CD4_Tcell,
  LAG3_IRF1_IFNrelated = LAG3_IRF1_IFNrelated
)



In [13]:
# Function to calculate mean expression for a gene set
calculate_signature_score <- function(seurat_obj, gene_list, signature_name) {
  # Convert gene list to vector
  genes <- unlist(gene_list)
  
  # Get genes that are present in the dataset
  available_genes <- genes[genes %in% rownames(seurat_obj)]
  
  cat("Signature:", signature_name, "\n")
  cat("  Total genes in signature:", length(genes), "\n")
  cat("  Genes found in dataset:", length(available_genes), "\n")
  
  # Get expression data 
  expr_data <- GetAssayData(seurat_obj, assay = "SCT", layer = "data")
  
  # Calculate mean expression across the available genes for each cell
  signature_scores <- colMeans(expr_data[available_genes, ], na.rm = TRUE)
  
  return(signature_scores)
}

In [ ]:
### Calculate scores for each signature and add to individual sample files
# Create analysed folder
analysed_folder <- file.path(res_folder, "analysed")
dir.create(analysed_folder, showWarnings = FALSE, recursive = TRUE)

cat("Saving analysed files to:", analysed_folder, "\n\n")

# Loop through each sample file
for(rds_file in rds_files){
  sample_name <- sub("^processed_", "", sub("\\.rds$", "", basename(rds_file)))
  cat("=== Processing sample:", sample_name, "===\n")
  
  # Load the sample
  sample_obj <- readRDS(rds_file)
  cat("  Loaded sample with", ncol(sample_obj), "spots\n")
  
  # Calculate and add each signature score
  for(sig_name in names(signatures)) {
    cat("  Calculating", sig_name, "signature...\n")
    
    scores <- calculate_signature_score(sample_obj, 
                                         signatures[[sig_name]], 
                                         sig_name)
    
    # Add as metadata with descriptive name
    metadata_col_name <- paste0(sig_name, "_mean")
    sample_obj <- AddMetaData(object = sample_obj,
                               metadata = scores,
                               col.name = metadata_col_name)
    
    cat("    Added", metadata_col_name, "to metadata\n")
    
    # Calculate AUC for this signature (skip for frozen_b_7)
    if(sample_name != "frozen_b_7") {
      tls_labels <- sample_obj$TLS_2_cat_annot == "TLS"
      sig_scores <- scores
      
      # Remove NA values
      valid_idx <- !is.na(sig_scores) & !is.na(tls_labels)
      
      # Calculate ROC and AUC
      roc_obj <- rocit(score = sig_scores[valid_idx], 
                      class = tls_labels[valid_idx], 
                      negref = FALSE)
      auc_value <- roc_obj$AUC
      
      # Add AUC as sample-level metadata (same value for all spots)
      auc_col_name <- paste0(sig_name, "_AUC")
      sample_obj[[auc_col_name]] <- auc_value
    }
  }
  
  # Save the updated object to analysed folder with analysed_ prefix
  analysed_file <- file.path(analysed_folder, paste0("analysed_", sample_name, ".rds"))
  saveRDS(sample_obj, analysed_file)
  cat("  Saved analysed sample to:", analysed_file, "\n")
  
  # Clear memory
  rm(sample_obj)
  gc(verbose = FALSE)
  
  cat("Finished processing", sample_name, "\n\n")
}



### Plotting

In [ ]:
###Generate plot panels for each sample
# Create plots folder
plots_folder <- file.path(res_folder, "plots")
dir.create(plots_folder, showWarnings = FALSE, recursive = TRUE)

# Get all analysed RDS files
analysed_folder <- file.path(res_folder, "analysed")
rds_files <- list.files(path = analysed_folder, 
                        pattern = "^analysed_.*\\.rds$", 
                        full.names = TRUE)

cat("Found", length(rds_files), "analysed sample files\n")
cat("Saving plot files to:", plots_folder, "\n\n")

# Define signature list
sign_list <- c("Literature_mean","CD4_LGMN_Treg_mean", "CXCR4_CD4_Tcell_mean", "LAG3_IRF1_IFNrelated_mean")

# Helper to hide axes
hide_axis <- theme(
  axis.title = element_blank(),
  axis.text = element_blank(),
  axis.ticks = element_blank()
)

# Loop through each sample file
for(rds_file in rds_files){
  sample_name <- sub("^analysed_", "", sub("\\.rds$", "", basename(rds_file)))
  cat("\n=== Creating plots for sample:", sample_name, "===\n")
  
  # Load sample
  sample_obj <- readRDS(rds_file)
  
  # Verify signature columns are present
  missing_sigs <- sign_list[!sign_list %in% colnames(sample_obj@meta.data)]
  if(length(missing_sigs) > 0) {
    cat("  WARNING: Missing signature columns:", paste(missing_sigs, collapse = ", "), "\n")
    cat("  Skipping sample - please re-run the analysis cell to regenerate this file.\n")
    rm(sample_obj); gc(verbose = FALSE)
    next
  }
  
  # Calculate individual percentile limits for this sample
  individual_limits <- list()
  for(sign in sign_list) {
    values <- sample_obj@meta.data[[sign]]
    individual_limits[[sign]] <- quantile(values, probs = c(0.05, 0.95), na.rm = TRUE)
    cat("  ", sign, ": [", individual_limits[[sign]][1], ",", individual_limits[[sign]][2], "]\n")
  }
  
  # H&E image plot
  he <- SpatialPlot(sample_obj, repel = FALSE, label = FALSE, 
                    image.alpha = 1, alpha = c(0, 0), pt.size.factor = 0) + 
    NoLegend()
  
  # TLS annotation plot 

  tls_annot_plot <- SpatialPlot(sample_obj, image.alpha = 0, 
                                group.by = "TLS_2_cat_annot",
                                pt.size.factor = 1.5) +
    ggtitle("TLS Annotation") +
    hide_axis +
    theme(legend.position = "right",
          legend.title = element_text(size = 12),
          plot.title = element_text(size = 14, hjust = 0.5))
  
  # Blank plot
  blank <- SpatialPlot(sample_obj, repel = FALSE, label = FALSE, 
                       image.alpha = 0, alpha = c(0, 0), pt.size.factor = 0) + 
    NoLegend()
  
  # Create plots for each signature with individual percentile-based limits
  all_plots <- lapply(sign_list, function(sign){
    p <- SpatialPlot(sample_obj, image.alpha = 0, features = sign) +
      scale_fill_gradientn(
        colors = c("blue", "cyan", "green", "yellow", "orange", "red"),
        limits = individual_limits[[sign]],
        oob = scales::squish  # Squish values outside limits instead of setting to NA
      ) +
      hide_axis + 
      theme(legend.title = element_text(size = 15),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank())
    return(p)
  })
  
  # Create AUC table 
  auc_data <- data.frame(
    Signature = character(),
    AUC = character(),
    stringsAsFactors = FALSE
  )
    
  for(sign in sign_list){
    sig_name_base <- gsub("_mean$", "", sign)
    auc_col <- paste0(sig_name_base, "_AUC")
    
    if(auc_col %in% colnames(sample_obj@meta.data)){
      auc_value <- sample_obj@meta.data[[auc_col]][1]
      auc_data <- rbind(auc_data, data.frame(
        Signature = sig_name_base,
        AUC = sprintf("%.3f", auc_value),
        stringsAsFactors = FALSE
      ))
    }
  }
  
  # Create table plot
  if(nrow(auc_data) > 0){
    table_grob <- tableGrob(auc_data, rows = NULL, 
                            theme = ttheme_minimal(
                              core = list(fg_params = list(hjust = 0, x = 0.1, fontsize = 12)),
                              colhead = list(fg_params = list(fontsize = 14, fontface = "bold"))
                            ))
    table_plot <- ggplot() + 
      annotation_custom(table_grob) + 
      theme_void()
    cat("  Created AUC table with", nrow(auc_data), "signatures\n")
  } else {
    table_plot <- ggplot() + theme_void()
  }

  
  # Combine plots: 3x3 layout
  combined_plot <- (he | all_plots[[1]] | all_plots[[2]]) / 
                 (all_plots[[3]] | all_plots[[4]] | tls_annot_plot) /
                 (table_plot | blank | blank)
  
  # Save PDF
  pdf_file <- file.path(plots_folder, paste0(sample_name, "_TLS_signatures.pdf"))
  pdf(pdf_file, width = 15, height = 15)
  print(combined_plot)
  dev.off()
  
  cat("  Saved plot to:", pdf_file, "\n")
  
  
  cat("Finished processing", sample_name, "\n")
}

cat("\n=== All plots saved to:", plots_folder, "===\n")


Found 5 analysed sample files
Saving plot files to: /lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/results/plots 




=== Creating plots for sample: analysed_ffpe_c_51 ===
  Skipping sample - please re-run the analysis cell to regenerate this file.

=== Creating plots for sample: analysed_frozen_b_1 ===
  Skipping sample - please re-run the analysis cell to regenerate this file.

=== Creating plots for sample: ffpe_c_51 ===
   Literature_mean : [ 0.7082218 , 1.772856 ]
   CD4_LGMN_Treg_mean : [ 0.255589 , 0.5640507 ]
   COL_LAIR_Immunosuppresive_mean : [ 0.5547174 , 1.062643 ]
   CXCR4_CD4_Tcell_mean : [ 0.3706381 , 0.7434823 ]
   LAG3_IRF1_IFNrelated_mean : [ 0.216769 , 0.5042524 ]


Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


  Created AUC table with 5 signatures
  Saved plot to: /lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/results/plots/ffpe_c_51_TLS_signatures.pdf 
Finished processing ffpe_c_51 

=== Creating plots for sample: frozen_b_1 ===
   Literature_mean : [ 0.7723572 , 2.279124 ]
   CD4_LGMN_Treg_mean : [ 0.5761698 , 1.068548 ]
   COL_LAIR_Immunosuppresive_mean : [ 0.5233803 , 1.035402 ]
   CXCR4_CD4_Tcell_mean : [ 0.2937802 , 0.5837912 ]
   LAG3_IRF1_IFNrelated_mean : [ 0.363059 , 0.6281726 ]


Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


  Created AUC table with 5 signatures
  Saved plot to: /lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/results/plots/frozen_b_1_TLS_signatures.pdf 
Finished processing frozen_b_1 

=== Creating plots for sample: frozen_b_7 ===
   Literature_mean : [ 0.3163561 , 0.5903156 ]
   CD4_LGMN_Treg_mean : [ 0.4614315 , 0.7946598 ]
   COL_LAIR_Immunosuppresive_mean : [ 0.3060395 , 0.597938 ]
   CXCR4_CD4_Tcell_mean : [ 0.2336991 , 0.459584 ]
   LAG3_IRF1_IFNrelated_mean : [ 0.4145304 , 0.7674201 ]


Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


  Saved plot to: /lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/results/plots/frozen_b_7_TLS_signatures.pdf 
Finished processing frozen_b_7 

=== All plots saved to: /lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/results/plots ===


# Hide

In [ ]:
###Generate plot panels for each sample
# Create plots folder
plots_folder <- file.path(res_folder, "plots")
dir.create(plots_folder, showWarnings = FALSE, recursive = TRUE)

# Get all analysed RDS files
analysed_folder <- file.path(res_folder, "analysed")
rds_files <- list.files(path = analysed_folder, 
                        pattern = "^analysed_.*\\.rds$", 
                        full.names = TRUE)

cat("Found", length(rds_files), "analysed sample files\n")
cat("Saving plot files to:", plots_folder, "\n\n")

# Define signature list
sign_list <- c("Literature_mean","Core1_mean", "Core2_mean", "TLS_border_mean", "TLS_border_2_mean")

# Calculate global min/max for each signature across all samples
cat("Calculating global min/max values for each signature...\n")
global_limits <- list()

for(sign in sign_list) {
  all_values <- c()
  
  for(rds_file in rds_files) {
    sample_obj <- readRDS(rds_file)
    
    if(sign %in% colnames(sample_obj@meta.data)) {
      all_values <- c(all_values, sample_obj@meta.data[[sign]])
    }
    
    rm(sample_obj)
    gc(verbose = FALSE)
  }
  
  global_limits[[sign]] <- c(min(all_values, na.rm = TRUE), 
                              max(all_values, na.rm = TRUE))
  cat("  ", sign, ": [", global_limits[[sign]][1], ",", global_limits[[sign]][2], "]\n")
}

cat("\n")

# Helper to hide axes
hide_axis <- theme(
  axis.title = element_blank(),
  axis.text = element_blank(),
  axis.ticks = element_blank()
)

# Loop through each sample file
for(rds_file in rds_files){
  sample_name <- sub("^analysed_", "", sub("\\.rds$", "", basename(rds_file)))
  cat("\n=== Creating plots for sample:", sample_name, "===\n")
  
  # Load sample
  sample_obj <- readRDS(rds_file)
  
  # H&E image plot
  he <- SpatialPlot(sample_obj, repel = FALSE, label = FALSE, 
                    image.alpha = 1, alpha = c(0, 0), pt.size.factor = 0) + 
    NoLegend()
  
  # TLS annotation plot (skip for frozen_b_7)
  if(sample_name != "frozen_b_7") {
    tls_annot_plot <- SpatialPlot(sample_obj, image.alpha = 0, 
                                  group.by = "TLS_2_cat_annot",
                                  pt.size.factor = 1.5) +
      ggtitle("TLS Annotation") +
      hide_axis +
      theme(legend.position = "right",
            legend.title = element_text(size = 12),
            plot.title = element_text(size = 14, hjust = 0.5))
  } else {
    # Create blank plot for frozen_b_7
    tls_annot_plot <- SpatialPlot(sample_obj, repel = FALSE, label = FALSE, 
                                   image.alpha = 0, alpha = c(0, 0), pt.size.factor = 0) + 
      NoLegend()
  }
  
  # Blank plot
  blank <- SpatialPlot(sample_obj, repel = FALSE, label = FALSE, 
                       image.alpha = 0, alpha = c(0, 0), pt.size.factor = 0) + 
    NoLegend()
  
  # Create plots for each signature with global limits
  all_plots <- lapply(sign_list, function(sign){
    p <- SpatialPlot(sample_obj, image.alpha = 0, features = sign) +
      scale_fill_gradientn(
        colors = c("blue", "cyan", "green", "yellow", "orange", "red"),
        limits = global_limits[[sign]],
        oob = scales::squish  # Squish values outside limits instead of setting to NA
      ) +
      hide_axis + 
      theme(legend.title = element_text(size = 15),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank())
    return(p)
  })
  
  # Create AUC table (skip for frozen_b_7)
  if(sample_name != "frozen_b_7") {
    auc_data <- data.frame(
      Signature = character(),
      AUC = character(),
      stringsAsFactors = FALSE
    )
    
    for(sign in sign_list){
      sig_name_base <- gsub("_mean$", "", sign)
      auc_col <- paste0(sig_name_base, "_AUC")
      
      if(auc_col %in% colnames(sample_obj@meta.data)){
        auc_value <- sample_obj@meta.data[[auc_col]][1]
        auc_data <- rbind(auc_data, data.frame(
          Signature = sig_name_base,
          AUC = sprintf("%.3f", auc_value),
          stringsAsFactors = FALSE
        ))
      }
    }
    
    # Create table plot
    if(nrow(auc_data) > 0){
      table_grob <- tableGrob(auc_data, rows = NULL, 
                             theme = ttheme_minimal(
                               core = list(fg_params = list(hjust = 0, x = 0.1, fontsize = 12)),
                               colhead = list(fg_params = list(fontsize = 14, fontface = "bold"))
                             ))
      table_plot <- ggplot() + 
        annotation_custom(table_grob) + 
        theme_void()
      cat("  Created AUC table with", nrow(auc_data), "signatures\n")
    } else {
      table_plot <- ggplot() + theme_void()
    }
  } else {
    # Create blank table plot for frozen_b_7
    table_plot <- ggplot() + theme_void()
  }
  
  # Combine plots: 3x3 layout
  combined_plot <- (he | all_plots[[1]] | all_plots[[2]]) / 
                   (all_plots[[3]] | all_plots[[4]] | all_plots[[5]]) /
                   (tls_annot_plot | table_plot | blank)
  
  # Save PDF
  pdf_file <- file.path(plots_folder, paste0(sample_name, "_TLS_signatures.pdf"))
  pdf(pdf_file, width = 15, height = 15)
  print(combined_plot)
  dev.off()
  
  cat("  Saved plot to:", pdf_file, "\n")
  
  
  cat("Finished processing", sample_name, "\n")
}

cat("\n=== All plots saved to:", plots_folder, "===\n")